# `test_data_quality.py` — Unit Tests for `DataQualityChecker`

## Purpose

Validates every public method of `DataQualityChecker`, ensuring the class correctly loads data,
detects missing values, infers data types, counts duplicates, checks value ranges, and generates
a consolidated quality report — including proper error handling for bad inputs.

---

## Module Under Test

`src.data_quality.data_quality_checker.DataQualityChecker`

---

## Test Classes at a Glance

| Class | Methods Tested | What It Verifies |
|-------|----------------|-----------------|
| `TestDataQualityCheckerLoad` | `load_data()` | Successful load, stored instance, file-not-found error, empty file error |
| `TestMissingValues` | `check_missing_values()` | Clean data, injected NaNs, pre-condition guard |
| `TestDataTypes` | `check_data_types()` | Return type, column index completeness |
| `TestDuplicates` | `check_duplicates()` | No duplicates, detected duplicates |
| `TestValueRanges` | `check_value_ranges()` | All columns pass range validation, salary categories |
| `TestQualityReport` | `generate_quality_report()` | Required keys, row count, column count |

---

## Fixtures Used

| Fixture | Source |
|---------|--------|
| `sample_df` | `conftest.py` — 200-row synthetic DataFrame |
| `sample_csv_path` | `conftest.py` — temporary CSV file path |
| `tmp_path` | pytest built-in — isolated temporary directory |

---

## How to Run

```bash
pytest tests/test_data_quality.py -v
```


---

## `TestDataQualityCheckerLoad`

**Purpose:** Tests the `load_data()` method — the entry point for the entire data quality pipeline.
Verifies that valid files are loaded correctly, the result is stored on the instance, and
appropriate errors are raised for missing or empty files.

### Test Methods

| Method | Verifies |
|--------|----------|
| `test_load_data_success` | Returns a `pd.DataFrame` with 200 rows and a `left` column |
| `test_load_data_returns_dataframe_stored_on_instance` | `checker.df` is populated after load |
| `test_load_data_file_not_found` | Raises `DataLoadError` with "Dataset not found" message |
| `test_load_data_empty_file` | Raises `DataLoadError` when CSV is empty |


In [ ]:
import numpy as np
import pandas as pd
import pytest

from src.data_quality.data_quality_checker import DataQualityChecker
from src.utils.exceptions import DataLoadError, DataQualityError


class TestDataQualityCheckerLoad:
    def test_load_data_success(self, sample_csv_path):
        checker = DataQualityChecker(str(sample_csv_path))
        df = checker.load_data()
        assert isinstance(df, pd.DataFrame)
        assert len(df) == 200
        assert "left" in df.columns

    def test_load_data_returns_dataframe_stored_on_instance(self, sample_csv_path):
        checker = DataQualityChecker(str(sample_csv_path))
        df = checker.load_data()
        assert checker.df is not None
        assert len(checker.df) == len(df)

    def test_load_data_file_not_found(self, tmp_path):
        checker = DataQualityChecker(str(tmp_path / "nonexistent.csv"))
        with pytest.raises(DataLoadError, match="Dataset not found"):
            checker.load_data()

    def test_load_data_empty_file(self, tmp_path):
        empty = tmp_path / "empty.csv"
        empty.write_text("")
        checker = DataQualityChecker(str(empty))
        with pytest.raises(DataLoadError):
            checker.load_data()


---

## `TestMissingValues`

**Purpose:** Tests `check_missing_values()` — verifies both the happy path (clean data returns
zero missing counts) and the failure path (NaN-injected data is correctly detected). Also guards
that the method raises `DataQualityError` if called before `load_data()`.

### Test Methods

| Method | Verifies |
|--------|----------|
| `test_no_missing_values` | Result has `missing_count` and `missing_percentage` columns; sum is 0 |
| `test_with_missing_values` | 5 injected NaNs in `satisfaction_level` are correctly counted |
| `test_requires_load_data_first` | Raises `DataQualityError` matching "load_data" if data not loaded |


In [ ]:
class TestMissingValues:
    def test_no_missing_values(self, sample_csv_path):
        checker = DataQualityChecker(str(sample_csv_path))
        checker.load_data()
        result = checker.check_missing_values()
        assert "missing_count" in result.columns
        assert "missing_percentage" in result.columns
        assert result["missing_count"].sum() == 0

    def test_with_missing_values(self, sample_df, tmp_path):
        df_with_nan = sample_df.copy()
        df_with_nan.loc[0:4, "satisfaction_level"] = np.nan
        csv_path = tmp_path / "missing.csv"
        df_with_nan.to_csv(csv_path, index=False)
        checker = DataQualityChecker(str(csv_path))
        checker.load_data()
        result = checker.check_missing_values()
        assert result.loc["satisfaction_level", "missing_count"] == 5

    def test_requires_load_data_first(self, sample_csv_path):
        checker = DataQualityChecker(str(sample_csv_path))
        with pytest.raises(DataQualityError, match="load_data"):
            checker.check_missing_values()


---

## `TestDataTypes`

**Purpose:** Tests `check_data_types()` — confirms the method returns a well-formed DataFrame
with a `dtype` column and that every column in the source dataset is represented in the index.

### Test Methods

| Method | Verifies |
|--------|----------|
| `test_check_data_types_returns_dataframe` | Return type is `pd.DataFrame` with a `dtype` column |
| `test_check_data_types_has_all_columns` | Index of result matches all columns of `sample_df` |


In [ ]:
class TestDataTypes:
    def test_check_data_types_returns_dataframe(self, sample_csv_path):
        checker = DataQualityChecker(str(sample_csv_path))
        checker.load_data()
        result = checker.check_data_types()
        assert isinstance(result, pd.DataFrame)
        assert "dtype" in result.columns

    def test_check_data_types_has_all_columns(self, sample_csv_path, sample_df):
        checker = DataQualityChecker(str(sample_csv_path))
        checker.load_data()
        result = checker.check_data_types()
        assert set(result.index) == set(sample_df.columns)


---

## `TestDuplicates`

**Purpose:** Tests `check_duplicates()` — verifies that the method returns zero for clean data
and accurately counts duplicated rows when they are artificially injected.

### Test Methods

| Method | Verifies |
|--------|----------|
| `test_no_duplicates` | Returns `0` for the clean synthetic dataset |
| `test_with_duplicates` | Returns `10` when 10 duplicate rows are appended to the CSV |


In [ ]:
class TestDuplicates:
    def test_no_duplicates(self, sample_csv_path):
        checker = DataQualityChecker(str(sample_csv_path))
        checker.load_data()
        assert checker.check_duplicates() == 0

    def test_with_duplicates(self, sample_df, tmp_path):
        df_dup = pd.concat([sample_df, sample_df.iloc[:10]], ignore_index=True)
        csv_path = tmp_path / "dups.csv"
        df_dup.to_csv(csv_path, index=False)
        checker = DataQualityChecker(str(csv_path))
        checker.load_data()
        assert checker.check_duplicates() == 10


---

## `TestValueRanges`

**Purpose:** Tests `check_value_ranges()` — confirms that every column in the synthetic dataset
falls within its defined business rules (e.g., `satisfaction_level` ∈ [0,1], `salary` ∈ {low,
medium, high}) and that the result structure is correct.

### Test Methods

| Method | Verifies |
|--------|----------|
| `test_valid_ranges` | All columns pass their range check; `valid=True` for every entry |
| `test_salary_valid_values` | `salary` column reports `valid=True` and `unexpected_values=[]` |


In [ ]:
class TestValueRanges:
    def test_valid_ranges(self, sample_csv_path):
        checker = DataQualityChecker(str(sample_csv_path))
        checker.load_data()
        results = checker.check_value_ranges()
        for col, info in results.items():
            assert info["valid"], f"Column {col} failed range check: {info}"

    def test_salary_valid_values(self, sample_csv_path):
        checker = DataQualityChecker(str(sample_csv_path))
        checker.load_data()
        results = checker.check_value_ranges()
        assert results["salary"]["valid"]
        assert results["salary"]["unexpected_values"] == []


---

## `TestQualityReport`

**Purpose:** Tests `generate_quality_report()` — validates the consolidated report dictionary
contains all required keys and that summary statistics (row count, column count) are accurate.

### Test Methods

| Method | Verifies |
|--------|----------|
| `test_report_has_all_keys` | Report contains all 7 expected keys |
| `test_report_total_rows` | `total_rows` equals 200 |
| `test_report_total_columns` | `total_columns` matches `len(sample_df.columns)` |


In [ ]:
class TestQualityReport:
    def test_report_has_all_keys(self, sample_csv_path):
        checker = DataQualityChecker(str(sample_csv_path))
        checker.load_data()
        report = checker.generate_quality_report()
        expected_keys = {
            "missing_values",
            "data_types",
            "duplicate_count",
            "value_ranges",
            "total_rows",
            "total_columns",
            "column_names",
        }
        assert expected_keys.issubset(set(report.keys()))

    def test_report_total_rows(self, sample_csv_path):
        checker = DataQualityChecker(str(sample_csv_path))
        checker.load_data()
        report = checker.generate_quality_report()
        assert report["total_rows"] == 200

    def test_report_total_columns(self, sample_csv_path, sample_df):
        checker = DataQualityChecker(str(sample_csv_path))
        checker.load_data()
        report = checker.generate_quality_report()
        assert report["total_columns"] == len(sample_df.columns)
